In [18]:
import pandas as pd
import numpy as np
import tensorflow as tf
import time
import tracemalloc
import heapq
import dlx  # pip install dlx
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

In [19]:

# 1) DATA LOADING & PREPROCESSING

def string_to_grid(s):
    s = s.strip()
    if len(s) != 81:
        raise ValueError(f"Expected 81 chars, got {len(s)}")
    return np.array([int(c) for c in s]).reshape(9, 9)

# load
df = pd.read_csv("/kaggle/input/sudoku-dataset/sudoku.csv")
df = df.dropna(subset=['quizzes','solutions'])
df['quizzes']   = df['quizzes'].astype(str)
df['solutions'] = df['solutions'].astype(str)

# X: (N,9,9,1), y: (N,81)
X = np.array([string_to_grid(q) for q in df['quizzes']])[..., np.newaxis]
y = (np.array([string_to_grid(s) for s in df['solutions']]) - 1).reshape(-1, 81)

# split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# tf.data pipeline
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(len(X_train))
    .batch(64)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(64)
    .prefetch(tf.data.AUTOTUNE)
)

/tmp/ipykernel_31/2955359566.py:11: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/kaggle/input/sudoku-dataset/sudoku.csv")


In [20]:

# 2) BUILD & TRAIN SHALLOW MLP

def build_sudoku_mlp():
    inp = layers.Input((9,9,1))
    x = layers.Flatten()(inp)                     # → 81
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(81*9, activation='relu')(x)  # → 729
    x = layers.Reshape((81, 9))(x)                # → (81,9)
    out = layers.Activation('softmax')(x)
    m = models.Model(inp, out)
    m.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return m

mlp = build_sudoku_mlp()
mlp.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)],
    verbose=2
)

Epoch 1/20


W0000 00:00:1745197826.468966     111 assert_op.cc:38] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
W0000 00:00:1745197842.915507     110 assert_op.cc:38] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert
W0000 00:00:1745197844.348329     111 assert_op.cc:38] Ignoring Assert operator compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/assert_equal_1/Assert/Assert


7500/7500 - 27s - 4ms/step - accuracy: 0.1244 - loss: 2.1818 - val_accuracy: 0.1272 - val_loss: 2.1722
Epoch 2/20
7500/7500 - 20s - 3ms/step - accuracy: 0.1274 - loss: 2.1716 - val_accuracy: 0.1293 - val_loss: 2.1640
Epoch 3/20
7500/7500 - 21s - 3ms/step - accuracy: 0.1290 - loss: 2.1649 - val_accuracy: 0.1300 - val_loss: 2.1605
Epoch 4/20
7500/7500 - 20s - 3ms/step - accuracy: 0.1298 - loss: 2.1619 - val_accuracy: 0.1312 - val_loss: 2.1574
Epoch 5/20
7500/7500 - 21s - 3ms/step - accuracy: 0.1310 - loss: 2.1587 - val_accuracy: 0.1319 - val_loss: 2.1554
Epoch 6/20
7500/7500 - 20s - 3ms/step - accuracy: 0.1319 - loss: 2.1564 - val_accuracy: 0.1329 - val_loss: 2.1526
Epoch 7/20
7500/7500 - 20s - 3ms/step - accuracy: 0.1325 - loss: 2.1543 - val_accuracy: 0.1337 - val_loss: 2.1505
Epoch 8/20
7500/7500 - 21s - 3ms/step - accuracy: 0.1332 - loss: 2.1523 - val_accuracy: 0.1339 - val_loss: 2.1493
Epoch 9/20
7500/7500 - 20s - 3ms/step - accuracy: 0.1341 - loss: 2.1506 - val_accuracy: 0.1350 - va

In [21]:

# 3) HEURISTICS EXTRACTION

def get_heuristics(grid):
    """
    Given a 9×9 grid with zeros for blanks, return:
      - probs: shape (81,9) per-cell digit probabilities
      - ent:   shape (81,) per-cell entropy
      - triple_probs: length = number of (r,c,d) rows for DLX, mapping each row to its p
      - rows_list: list of (r,c,d) triples in the same order as triple_probs
    """
    flat = grid[np.newaxis,...,np.newaxis]  # (1,9,9,1)
    probs = mlp.predict(flat)[0]            # (81,9)
    # entropy per cell
    ent = -np.sum(np.where(probs>0, probs * np.log(probs), 0), axis=1)

    # build rows_list and triple_probs for DLX
    rows_list = []
    for r in range(9):
        for c in range(9):
            if grid[r,c] != 0:
                rows_list.append((r,c,grid[r,c]-1))
            else:
                for d in range(9):
                    rows_list.append((r,c,d))
    # triple_probs: probability for that (cell, digit)
    triple_probs = [probs[r*9+c, d] for (r,c,d) in rows_list]
    return probs, ent, rows_list, triple_probs


In [22]:

# 4) SOLVER IMPLEMENTATIONS (all use heuristics)


def solve_bt_mlp(grid, probs, ent):
    """Backtracking guided by CNN: pick lowest-entropy empty cell, try values by descending prob."""
    nodes = {'count':0}
    def select_cell(g):
        empties = [i for i in range(81) if g.flat[i]==0]
        return min(empties, key=lambda i: ent[i]) if empties else None
    def backtrack(g):
        nodes['count']+=1
        ci = select_cell(g)
        if ci is None:
            return True
        i,j = divmod(ci,9)
        for d in np.argsort(-probs[ci]):
            v = d+1
            if v in g[i] or v in g[:,j]: continue
            br,bc = 3*(i//3),3*(j//3)
            if v in g[br:br+3,bc:bc+3]: continue
            g[i,j]=v
            if backtrack(g): return True
            g[i,j]=0
        return False

    g = grid.copy()
    tracemalloc.start(); t0=time.time()
    backtrack(g)
    tm = time.time()-t0; peak=tracemalloc.get_traced_memory()[1]; tracemalloc.stop()
    return g, nodes['count'], tm, peak/1024**2
    
def solve_astar_mlp(grid, probs):
    """A* with h(n)=#empty, use MLP for value‐ordering & tie‐break."""
    nodes = 0

    def h(g): return np.count_nonzero(g == 0)
    def g_cost(g): return -np.count_nonzero(g != 0)

    start = grid.copy()
    open_set = [(h(start) + g_cost(start), g_cost(start), start)]
    seen = {start.tobytes(): g_cost(start)}

    tracemalloc.start(); t0 = time.time()
    while open_set:
        _, _, s = heapq.heappop(open_set)
        nodes += 1
        if h(s) == 0:
            tm = time.time() - t0
            peak = tracemalloc.get_traced_memory()[1]
            tracemalloc.stop()
            return s, nodes, tm, peak / 1024**2

        # collect empties
        empties = [(i, j) for i in range(9) for j in range(9) if s[i, j] == 0]

        # pick MRV, tie‐break by highest CNN confidence
        best = None  # ( (domain_size, -max_conf), i, j, domain_list )
        for i, j in empties:
            # compute legal domain for (i,j)
            dom = [d+1 for d in range(9)
                   if d+1 not in s[i] and
                      d+1 not in s[:, j] and
                      d+1 not in s[3*(i//3):3*(i//3)+3, 3*(j//3):3*(j//3)+3].flatten()]
            if not dom:
                continue
            cell = i*9 + j
            # extract the max probability for that cell/domain
            maxp = max(probs[cell, d-1] for d in dom)
            score = (len(dom), -maxp)
            if best is None or score < best[0]:
                best = (score, i, j, dom)

        _, i, j, dom = best
        cell = i*9 + j

        # push children in order of descending confidence
        for d in sorted(dom, key=lambda d: -probs[cell, d-1]):
            new = s.copy()
            new[i, j] = d
            gc = -np.count_nonzero(new != 0)
            key = new.tobytes()
            if key not in seen or seen[key] < gc:
                seen[key] = gc
                heapq.heappush(open_set, (h(new) + gc, gc, new))

    # no solution
    tm = time.time() - t0
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    return None, nodes, tm, peak / 1024**2


def solve_csp_mlp(grid, probs):
    """CSP with MRV+FC; tie‐break MRV by CNN confidence; value‐order by prob."""
    nodes = {'count': 0}

    # initial domains
    doms = [[{grid[r, c]} if grid[r, c] != 0 else set(range(1, 10))
             for c in range(9)] for r in range(9)]

    def propagate(i, j, val, doms):
        new = [[doms[r][c].copy() for c in range(9)] for r in range(9)]
        for k in range(9):
            new[i][k].discard(val)
            new[k][j].discard(val)
        br, bc = 3*(i//3), 3*(j//3)
        for di in range(3):
            for dj in range(3):
                new[br+di][bc+dj].discard(val)
        return new

    def backtrack(d):
        nodes['count'] += 1
        # failure check
        for r in range(9):
            for c in range(9):
                if not d[r][c]:
                    return None
        # completion check
        if all(len(d[r][c]) == 1 for r in range(9) for c in range(9)):
            sol = np.zeros((9, 9), int)
            for r in range(9):
                for c in range(9):
                    sol[r, c] = next(iter(d[r][c]))
            return sol

        # MRV + CNN‐tie‐break
        best = None  # ( (domain_size, -max_conf), r, c )
        for r in range(9):
            for c in range(9):
                if len(d[r][c]) > 1:
                    cell = r*9 + c
                    # get max probability over this cell's domain
                    maxp = max(probs[cell, v-1] for v in d[r][c])
                    score = (len(d[r][c]), -maxp)
                    if best is None or score < best[0]:
                        best = (score, r, c)
        _, i, j = best

        # value‐order by descending prob
        for v in sorted(d[i][j], key=lambda v: -probs[i*9 + j, v-1]):
            nd = propagate(i, j, v, d)
            nd[i][j] = {v}
            res = backtrack(nd)
            if res is not None:
                return res
        return None

    tracemalloc.start(); t0 = time.time()
    sol = backtrack(doms)
    tm = time.time() - t0
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()

    return sol, nodes['count'], tm, peak / 1024**2


# Define the pure-Python Algorithm X solver guided by MLP heuristics
def solve_dlx_mlp_count(grid, rows_list, triple_probs):
    """
    Algorithm X (exact cover) guided by MLP heuristics (triple_probs).
    Returns: solution_grid, nodes_explored, time_s, peak_memory_MB
    """
    # 1) Build exact-cover matrix rows
    matrix = [
        [
            9*r + c,
            81 + 9*r + d,
            162 + 9*c + d,
            243 + 9*((r//3)*3 + c//3) + d
        ]
        for (r, c, d) in rows_list
    ]

    # 2) Column -> rows mapping
    col_to_rows = {col: set() for col in range(324)}
    for i, cols in enumerate(matrix):
        for col in cols:
            col_to_rows[col].add(i)

    active_cols = set(range(324))
    active_rows = set(range(len(matrix)))
    node_count = 0

    def search(solution):
        nonlocal node_count, active_cols, active_rows
        node_count += 1
        if not active_cols:
            return solution.copy()

        # MRV: choose column with fewest active rows
        c = min(active_cols, key=lambda col: len(col_to_rows[col] & active_rows))
        rows_covering = list(col_to_rows[c] & active_rows)
        if not rows_covering:
            return None

        # Order rows by descending heuristic probability
        rows_covering.sort(key=lambda r: -triple_probs[r])

        for r in rows_covering:
            solution.append(r)
            cols_covered = matrix[r]

            # Save state
            saved_rows = active_rows.copy()
            saved_cols = active_cols.copy()

            # Remove covered rows and cols
            rows_to_remove = set()
            for col in cols_covered:
                rows_to_remove |= (col_to_rows[col] & active_rows)
            active_rows -= rows_to_remove
            active_cols -= set(cols_covered)

            # Recurse
            result = search(solution)
            if result is not None:
                return result

            # Restore state
            active_rows = saved_rows
            active_cols = saved_cols
            solution.pop()

        return None

    # Run search with timing & memory
    tracemalloc.start()
    t0 = time.time()
    chosen = search([])
    t1 = time.time()
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    # Build solution grid if found
    solution_grid = None
    if chosen:
        solution_grid = np.zeros((9, 9), int)
        for idx in chosen:
            r, c, d = rows_list[idx]
            solution_grid[r, c] = d + 1

    return solution_grid, node_count, (t1 - t0), peak / 1024**2


In [23]:

# 5) RUN ALL SOLVERS ON AN EXAMPLE PUZZLE

if __name__ == "__main__":
    puzzle = np.array([
        [5,3,0,0,7,0,0,0,0],
        [6,0,0,1,9,5,0,0,0],
        [0,9,8,0,0,0,0,6,0],
        [8,0,0,0,6,0,0,0,3],
        [4,0,0,8,0,3,0,0,1],
        [7,0,0,0,2,0,0,0,6],
        [0,6,0,0,0,0,2,8,0],
        [0,0,0,4,1,9,0,0,5],
        [0,0,0,0,8,0,0,7,9]
    ])

    probs, ent, rows_list, triple_probs = get_heuristics(puzzle)

    solvers = {
        "Backtracking": lambda: solve_bt_mlp(puzzle, probs, ent),
        "A* Search":    lambda: solve_astar_mlp(puzzle, probs),
        "CSP":           lambda: solve_csp_mlp(puzzle, probs),
        "DLX":           lambda: solve_dlx_mlp_count(puzzle, rows_list, triple_probs)
    }

    for name, fn in solvers.items():
        sol, nodes, tm, mem = fn()
        print(f"\n=== {name} with MLP Heuristics ===")
        print(f"Nodes generated : {nodes}")
        print(f"Time (s)        : {tm:.4f}")
        print(f"Peak mem (MB)   : {mem:.2f}")
        print("Solution grid:")
        print(sol)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 310ms/step

=== Backtracking with MLP Heuristics ===
Nodes generated : 5984
Time (s)        : 2.8817
Peak mem (MB)   : 2239.19
Solution grid:
[[5 3 4 6 7 8 9 1 2]
 [6 7 2 1 9 5 3 4 8]
 [1 9 8 3 4 2 5 6 7]
 [8 5 9 7 6 1 4 2 3]
 [4 2 6 8 5 3 7 9 1]
 [7 1 3 9 2 4 8 5 6]
 [9 6 1 5 3 7 2 8 4]
 [2 8 7 4 1 9 6 3 5]
 [3 4 5 2 8 6 1 7 9]]

=== A* Search with MLP Heuristics ===
Nodes generated : 52
Time (s)        : 0.3538
Peak mem (MB)   : 0.04
Solution grid:
[[5 3 4 6 7 8 9 1 2]
 [6 7 2 1 9 5 3 4 8]
 [1 9 8 3 4 2 5 6 7]
 [8 5 9 7 6 1 4 2 3]
 [4 2 6 8 5 3 7 9 1]
 [7 1 3 9 2 4 8 5 6]
 [9 6 1 5 3 7 2 8 4]
 [2 8 7 4 1 9 6 3 5]
 [3 4 5 2 8 6 1 7 9]]

=== CSP with MLP Heuristics ===
Nodes generated : 1957
Time (s)        : 0.6022
Peak mem (MB)   : 1.36
Solution grid:
[[5 3 4 6 7 8 9 1 2]
 [6 7 2 1 9 5 3 4 8]
 [1 9 8 3 4 2 5 6 7]
 [8 5 9 7 6 1 4 2 3]
 [4 2 6 8 5 3 7 9 1]
 [7 1 3 9 2 4 8 5 6]
 [9 6 1 5 3 7 2 8 4]
 [2 8 7 4 1 9 6 3 5]
 [3 4 5 2 8 6 1 7 9]]

=== DLX with MLP 